# Inference-Time Scaling & Verifiers — AI Lab Instructor Notebook

*LLM & Agentic AI · Expert*



**Outcome.** You can simulate repeated sampling, compare pass@1 vs best-of-N, and measure how verifiers change final accuracy.

7 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f"Failed: {name}")
    print(f"OK: {name}")

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Inference-Time Scaling & Verifiers — Student Lab

Offline lab: simulate repeated sampling, exact verification, and noisy judges with a fixed stochastic model.

This lab studies a core idea in modern reasoning systems: even when model weights are fixed, we can spend more inference-time compute by sampling multiple candidates and selecting among them.

You will implement:

- temperature-based sampling from a stochastic candidate bank
- exact verification and best-of-N selection
- pass@1 vs coverage comparisons
- diversity / accuracy tradeoffs from temperature
- judge vs verifier selection quality
- adaptive test-time compute allocation by difficulty

## 0 — Toy stochastic problem bank

In [ ]:
$10

**Why this works.** # Inference-Time Scaling & Verifiers — Instructor Lab

Offline lab: simulate repeated sampling, exact verification, and noisy judges with a fixed stochastic model.

## 0 — Toy stochastic problem bank

## Task 2: 0 — Sample from the stochastic reasoner

## 0 — Sample from the stochastic reasoner

Implement sampling from the toy candidate bank.

Requirements:

- convert candidate logits into probabilities using temperature
- subtract the maximum logit before exponentiating for numerical stability
- sample one candidate using the provided random generator

The goal of this section is to show that inference-time scaling depends on stochasticity: if the model can produce different candidates, extra samples can improve coverage.

In [ ]:
def sample_candidate(problem: dict, temperature: float, rng: np.random.Generator):
    logits = np.array([c['logit'] for c in problem['candidates']], dtype=np.float64) / float(temperature)
    logits = logits - logits.max()
    probs = np.exp(logits)
    probs = probs / probs.sum()
    idx = int(rng.choice(len(problem['candidates']), p=probs))
    chosen = problem['candidates'][idx]
    return {k: chosen[k] for k in chosen}

cand = sample_candidate(PROBLEMS[0], temperature=1.0, rng=np.random.default_rng(0))
print('sample', cand['text'])
check('cand_has_text', isinstance(cand['text'], str) and len(cand['text']) > 0)

In [ ]:
# Checks
check('cand_has_text', isinstance(cand['text'], str) and len(cand['text']) > 0)

## Task 3: 1 — Exact verifier + best-of-N

## 1 — Exact verifier + best-of-N

Implement an exact verifier and a simple best-of-N policy.

Requirements:

- `verify` should check whether a sampled candidate exactly matches the gold answer
- `best_of_n` should sample `n_samples` candidates
- return the first verified-correct candidate if one exists
- otherwise fall back to the first sampled candidate

This section demonstrates the core best-of-N idea: the model may fail on one sample but succeed across multiple attempts if a strong verifier is available.

In [ ]:
def verify(problem: dict, cand: dict) -> bool:
    return str(cand['text']).strip() == str(problem['gold']).strip()

def best_of_n(problem: dict, n_samples: int, temperature: float, rng: np.random.Generator):
    sampled = [sample_candidate(problem, temperature=temperature, rng=rng) for _ in range(n_samples)]
    chosen = sampled[0]
    solved = False
    for cand in sampled:
        if verify(problem, cand):
            chosen = cand
            solved = True
            break
    return chosen, solved, sampled

chosen, solved, sampled = best_of_n(PROBLEMS[3], n_samples=8, temperature=1.1, rng=np.random.default_rng(0))
print('chosen', chosen['text'], 'solved', solved)
check('sample_count', len(sampled) == 8)

In [ ]:
# Checks
check('sample_count', len(sampled) == 8)

## Task 4: 2 — Pass@1 vs best-of-N coverage

## 2 — Pass@1 vs best-of-N coverage

Measure how success changes as we increase the number of samples.

Requirements:

- compute a success rate over the whole problem bank
- use repeated random trials for a stable estimate
- compare `n_samples` in `[1, 2, 4, 8]`

This section is about **coverage**: as the sample budget grows, the chance that at least one candidate is correct should increase.

In [ ]:
def success_rate(problem_bank, n_samples: int, temperature: float, trials: int = 200):
    vals = []
    for seed in range(trials):
        trial_rng = np.random.default_rng(seed)
        ok = 0
        for problem in problem_bank:
            _, solved, _ = best_of_n(problem, n_samples=n_samples, temperature=temperature, rng=trial_rng)
            ok += int(solved)
        vals.append(ok / len(problem_bank))
    return float(np.mean(vals))

rates = {n: success_rate(PROBLEMS, n_samples=n, temperature=1.1, trials=200) for n in [1, 2, 4, 8]}
print(rates)
check('mono', rates[1] <= rates[2] <= rates[4] <= rates[8])

In [ ]:
# Checks
check('mono', rates[1] <= rates[2] <= rates[4] <= rates[8])

## Task 5: 3 — Temperature tradeoff

## 3 — Temperature tradeoff

Measure the tradeoff between diversity and accuracy.

Requirements:

- estimate how many unique candidates appear under repeated sampling
- compare low, medium, and high temperatures
- measure both diversity and downstream success

This section shows why “sample more” is incomplete advice: extra samples help only if temperature creates useful diversity without making outputs too noisy.

In [ ]:
$11

In [ ]:
# Checks
check('high_more_diverse', div_high > div_low)
check('mid_beats_low_for_bestofn', acc_mid > acc_low)
check('mid_beats_high_pass1', acc_mid1 > acc_high1)

## Task 6: 4 — Judge vs verifier

## 4 — Judge vs verifier

Compare two selection mechanisms.

Requirements:

- `judge_select` should pick the sample with the highest judge score
- `compare_selectors` should measure final accuracy using both the judge and the exact verifier
- evaluate across many trials on the same problem bank

This section highlights a key production distinction: an exact verifier is usually much stronger than a plausibility-based judge when exact checking is possible.

In [ ]:
$12

In [ ]:
# Checks
check('verifier_ge_judge', verifier_acc > judge_acc)

## Task 7: 5 — Adaptive sampling budget

## 5 — Adaptive sampling budget

Allocate different inference-time compute budgets to easy, medium, and hard problems.

Requirements:

- implement a policy that uses smaller budgets for easier problems
- compare adaptive allocation against a fixed budget policy
- report both accuracy and average budget used

This section shows the engineering tradeoff behind reasoning systems: extra compute should be spent where it helps most, not uniformly on every query.

In [ ]:
$13

In [ ]:
# Checks
check('adaptive_uses_less_budget', budget_adapt < budget_fixed)
check('adaptive_not_too_much_worse', acc_adapt >= acc_fixed - 0.08)